# Business Understanding

## Data Preparation

Explicacion Tecnica y Metodologica

1. Que realizamos en estos dos pasos:

- Paso 1: Reduccion inteligente (Muestreo Estratificado). En lugar de usar los 25 millones de registros, extrajimos una muestra representativa del 5%. Para garantizar que fuera un reflejo exacto de la realidad, clasificamos a los usuarios por su nivel de actividad (baja, media y alta) y tomamos una proporcion igual de cada grupo. Ademas, descartamos las peliculas con menos de 20 valoraciones, ya que los algoritmos de recomendacion necesitan datos suficientes para encontrar patrones.

- Paso 2: Sincronizacion de metadatos. Al reducir el catalogo a unas 5,900 peliculas utiles, los archivos secundarios (como los generos y el genoma) quedaron desactualizados. Lo que hicimos fue filtrar estos archivos para conservar unicamente las caracteristicas de las peliculas que sobrevivieron al primer paso.

2. Por que se decidio usar el formato .parquet en lugar de .csv:

- Compresion de almacenamiento: Parquet reduce drasticamente el peso del archivo en el disco duro. Un CSV que pesa 1 GB puede reducirse a 100 MB sin perder un solo dato.
- Velocidad de lectura y consumo de RAM: Parquet es un formato "columnar". Esto permite que Python cargue partes especificas a la memoria RAM de manera instantanea, evitando que el entorno de Deepnote colapse por falta de memoria.
- Preservacion de tipos de datos: Cuando guardas un CSV, todo se convierte en texto. Parquet guarda la estructura exacta (enteros, decimales), ahorrando mucho tiempo de procesamiento computacional.

3. Por que es mejor utilizar una muestra reducida y menos archivos:

- Factibilidad tecnica: Los entornos en la nube con planes estudiantiles tienen limites de RAM. Intentar cruzar matrices con 25 millones de interacciones generaria un error de desbordamiento de memoria (Out of Memory).
- Velocidad de iteracion: El desarrollo de algoritmos requiere prueba y error. Entrenar un modelo con un millon de registros toma un par de minutos, permitiendo hacer correcciones rapidas.
- Reduccion de ruido estadistico: Al eliminar peliculas con pocas valoraciones, eliminamos datos que confunden al modelo. Retener solo los datos con volumen suficiente mejora significativamente la precision de las predicciones finales.

In [ ]:
import pandas as pd
from pathlib import Path
import gc

# 1. Rutas absolutas exactas en Deepnote
RUTA_RATINGS = Path('/work/data/raw/ratings.csv')
PROCESSED_DIR = Path('/work/data/intermediate')

# Crear la carpeta 'processed' si no existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Cargando el dataset original (esto puede tomar un minuto)...")
# Cargar solo las columnas necesarias para ahorrar memoria
df_ratings = pd.read_csv(
    RUTA_RATINGS, 
    usecols=['userId', 'movieId', 'rating', 'timestamp']
)

# 2. Conteo de actividad de usuarios para el Muestreo Estratificado
print("Calculando estratos de usuarios...")
user_counts = df_ratings['userId'].value_counts()
user_stats = pd.DataFrame({'userId': user_counts.index, 'num_ratings': user_counts.values})

# Dividir en 3 categorías (Casuales, Regulares, Cinéfilos)
user_stats['Tier'] = pd.qcut(user_stats['num_ratings'], q=3, labels=['Casuales', 'Regulares', 'Cinefilos'])

# Tomar una muestra del 5% (Muestreo estratificado)
fraccion_muestra = 0.05 
usuarios_seleccionados = user_stats.groupby('Tier', group_keys=False).apply(
    lambda x: x.sample(frac=fraccion_muestra, random_state=42)
)

# 3. Filtrar el dataset original
print("Aplicando el filtro de usuarios...")
df_muestra = df_ratings[df_ratings['userId'].isin(usuarios_seleccionados['userId'])]

# Liberar memoria RAM del dataset gigante original
del df_ratings
gc.collect()

# 4. Reducir el "Cold Start": Eliminar películas con menos de 20 valoraciones
print("Filtrando películas poco frecuentes...")
movie_counts = df_muestra['movieId'].value_counts()
peliculas_populares = movie_counts[movie_counts >= 20].index
df_final = df_muestra[df_muestra['movieId'].isin(peliculas_populares)]

# 5. Guardar el resultado en formato Parquet
ruta_salida = PROCESSED_DIR / 'ratings_sample_5pct.parquet'
print(f"Guardando el dataset final en: {ruta_salida}")

# Guardar como Parquet 
df_final.to_parquet(ruta_salida, index=False)

print("\n--- RESUMEN ---")
print(f"Usuarios únicos: {df_final['userId'].nunique()}")
print(f"Películas únicas: {df_final['movieId'].nunique()}")
print(f"Total de valoraciones: {len(df_final)}")
print("¡Proceso completado con éxito!")

In [ ]:
import pandas as pd
from pathlib import Path

# Definir rutas
RAW_DIR = Path('/work/data/raw')
INTERIM_DIR = Path('/work/data/intermediate')

print("1. Cargando la lista de películas válidas...")
# Leemos solo la columna movieId de nuestro parquet procesado para ahorrar RAM
df_ratings = pd.read_parquet(INTERIM_DIR / 'ratings_sample_5pct.parquet', columns=['movieId'])
peliculas_validas = df_ratings['movieId'].unique()
print(f"Total de películas a conservar: {len(peliculas_validas)}")

print("\n2. Filtrando movies.csv...")
df_movies = pd.read_csv(RAW_DIR / 'movies.csv')
df_movies_filtrado = df_movies[df_movies['movieId'].isin(peliculas_validas)]
ruta_movies = INTERIM_DIR / 'movies_sample.parquet'
df_movies_filtrado.to_parquet(ruta_movies, index=False)
print(f"Guardado en: {ruta_movies}")

print("\n3. Filtrando genome-scores.csv (esto puede tomar unos segundos)...")
df_genome = pd.read_csv(RAW_DIR / 'genome-scores.csv')
df_genome_filtrado = df_genome[df_genome['movieId'].isin(peliculas_validas)]
ruta_genome = INTERIM_DIR / 'genome_scores_sample.parquet'
df_genome_filtrado.to_parquet(ruta_genome, index=False)
print(f"Guardado en: {ruta_genome}")

# Copiamos genome-tags.csv tal cual, ya que es solo un catálogo pequeño de 1,129 filas
print("\n4. Copiando catálogo de tags...")
df_tags = pd.read_csv(RAW_DIR / 'genome-tags.csv')
df_tags.to_parquet(INTERIM_DIR / 'genome_tags.parquet', index=False)

print("\n¡Todos los metadatos han sido sincronizados y guardados en formato Parquet!")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4ca4d80d-8b01-4cbd-afc3-93630d98a14a' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>